# HDSoC Waveform Packet Window Positions

For each waveform in each event, find which packet `window_position` values contributed to the waveform trace.

Packets are matched by **both channel and trigger timestamp**, so periodic batches cannot mix different triggers. Trigger-time consistency is a grouping invariant, not an independent measurement of trigger jitter.


In [ ]:
# Edit this path, then run the notebook from the top.
# Absolute paths and ~ are supported; relative paths start at the app directory.
DATA_FILE = ".artifacts/bank_format_verification/run121479/output.root"


In [ ]:
from pathlib import Path
import sys, os
# Works when Jupyter starts in the repository or a notebook subdirectory.
APP_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                if (p / 'notebooks/notebook_support.py').is_file())
if str(APP_ROOT / 'notebooks') not in sys.path:
    sys.path.insert(0, str(APP_ROOT / 'notebooks'))
from notebook_support import (open_data, present, hit_waveform, hit_baseline,
    hit_amplitude, hit_peak, waveform_entries as find_waveform_entries,
    sampic_unit, integer, waveform_packets, sampic_event_bytes)
import ROOT
r = ROOT
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
root_file, tree, DATA_FILE = open_data('HDSoC', DATA_FILE)
f = file = root_file
plt.rcParams.update({'figure.figsize': (12, 5), 'axes.grid': True, 'grid.alpha': 0.25})


In [ ]:
# Setup is performed above.


In [ ]:
# Setup is performed above.


In [ ]:
def waveform_window_positions(evt, wf):
    return [int(p.window_position) for p in waveform_packets(evt, wf)]


In [ ]:
# First: print one list of window positions for one waveform
event_index = 0
waveform_index = 0

tree.GetEntry(event_index)
evt = tree.nalu_event
wfs = list(evt.waveforms.waveforms)

if waveform_index >= len(wfs):
    raise IndexError(f"waveform_index={waveform_index} out of range for event {event_index}")

wf = wfs[waveform_index]
positions = waveform_window_positions(evt, wf)
print(f"Event {event_index}, waveform {waveform_index}, channel {integer(wf.channel_num)}")
print("Window positions:", positions)


In [ ]:
# Then: histogram of window position frequency across all events, one histogram per channel
window_pos_counts_by_channel = defaultdict(Counter)

for entry in range(tree.GetEntries()):
    tree.GetEntry(entry)
    evt = tree.nalu_event

    for wf in evt.waveforms.waveforms:
        ch = integer(wf.channel_num)
        positions = waveform_window_positions(evt, wf)
        window_pos_counts_by_channel[ch].update(positions)

channels = sorted(window_pos_counts_by_channel.keys())
n_channels = len(channels)

fig, axes = plt.subplots(n_channels, 1, figsize=(10, 3 * n_channels), sharex=False)
if n_channels == 1:
    axes = [axes]

for ax, ch in zip(axes, channels):
    counter = window_pos_counts_by_channel[ch]
    xs = sorted(counter.keys())
    ys = [counter[x] for x in xs]

    ax.bar(xs, ys, width=0.8, edgecolor="black")
    ax.set_title(f"Channel {ch}: Window Position Frequency")
    ax.set_xlabel("window_position")
    ax.set_ylabel("Count")

plt.tight_layout()
plt.show()


In [ ]:
f.Close()
